# Question Answering with LangChain, OpenAI, and MultiQuery Retriever


<br>

## Intro

In this lab you'll build a Retrieval-Augmented Generation (RAG) chatbot backed by Elasticsearch and OpenAI, using LangChain's `MultiQueryRetriever` to improve retrieval quality.

**Why use a `MultiQueryRetriever`?** A single user question, phrased one way, might miss documents that use different wording for the same idea. [`MultiQueryRetriever`](https://api.python.langchain.com/en/latest/retrievers/langchain.retrievers.multi_query.MultiQueryRetriever.html) asks an LLM to rephrase the original question into several alternative versions, runs a separate search for each version, and merges the results — giving you a broader, more relevant set of documents than a single search would.

Here's how the workflow will look like:

- **Setup (once):** split a sample dataset of documents into passages (chunks), turn each passage into an embedding, and store them in a vector database (in this case, we'll use Elasticsearch).
- **At question time:** the `MultiQueryRetriever` generates a few variations of the question, searches Elasticsearch with each one, and combines the retrieved passages. Those passages are then passed to an LLM, which uses them as context to write the final answer.


And here's what to expect from this lab:

1. **Complete a guided demo** — You'll be given the overall structure and process to follow: connecting to Elasticsearch and OpenAI, loading and chunking a sample set of workplace documents, embedding and indexing them into a vector store, and wiring up a `MultiQueryRetriever` chain that expands a question into several variations before retrieving and answering. Some steps are already implemented for you, and you'll need to complete the rest (e.g. filling in missing parameters and logic) to get it fully working.
2. **Replicate it yourself with variations** — Then, you'll create **at least two new iterations** of the question-answering step (new questions and/or settings) to explore how `MultiQueryRetriever` behaves.

By the end of this lab, you'll understand how multi-query retrieval works and be able to apply it to your own RAG pipelines.

<br>

## Install and import dependencies

Uncomment and run the cells below to install the dependencies required for this notebook.

Tip: Use a virtual environment to keep this project's dependencies isolated from your system Python and other projects.

In [ ]:
# !pip install "langchain<0.3" "langchain-core<0.3" "langchain-community<0.3" "langchain-openai<0.2" "langchain-elasticsearch<0.3" "jq==1.12.0"

In [1]:
from getpass import getpass

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_elasticsearch import ElasticsearchStore
from langchain.retrievers.multi_query import MultiQueryRetriever

from langchain_community.document_loaders import JSONLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

## Connect to Elasticsearch

In this lab, you'll use Elasticsearch as the vector database and search engine that stores your documents and retrieves the most relevant ones in response to user queries.

**What is Elasticsearch?**

Elasticsearch is a distributed search and analytics engine designed for fast retrieval over large collections of data. In AI applications, it can store both text and vector embeddings, making it well suited for Retrieval-Augmented Generation (RAG) workflows where an LLM retrieves relevant context before generating a response.


**Getting started with Elastic Cloud:**

1. If you don't already have one, sign up for a free trial: [https://www.elastic.co/](https://www.elastic.co/).
2. During sign-up (or afterwards), create a deployment/project. Either a classic deployment or a serverless project will work for this lab.
3. Once your deployment/project is ready, you'll need to find your **Cloud ID** and create an **API key**. 

<br>

> To find your **Cloud ID** and create an **API key**, 
> follow the instructions on the link below.
> 
>   👇👇👇
>
> https://www.elastic.co/search-labs/tutorials/install-elasticsearch/find-cloud-id-create-api-keys 📌
>
<br>

In [2]:
#
#
# To find your Cloud ID and create an API key, 
# follow the instructions on the link below:
#
# 👇👇👇
#
# https://www.elastic.co/search-labs/tutorials/install-elasticsearch/find-cloud-id-create-api-keys 📌
#
#
ELASTIC_CLOUD_ID = getpass("Elastic Cloud ID: ")
ELASTIC_API_KEY = getpass("Elastic Api Key: ")

# https://platform.openai.com/api-keys
OPENAI_API_KEY = getpass("OpenAI API key: ")

embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

vectorstore = ElasticsearchStore(
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
    index_name=None, #give it a meaningful name
    embedding=embeddings,
)

## Indexing Data into Elasticsearch
Let's download the sample dataset and deserialize the document.

In [3]:
from urllib.request import urlopen
import json

url = "https://raw.githubusercontent.com/elastic/elasticsearch-labs/main/example-apps/chatbot-rag-app/data/data.json"

response = urlopen(url)
data = json.load(response)

with open("temp.json", "w") as json_file:
    json.dump(data, json_file)

### Split Documents into Passages

We’ll chunk documents into passages in order to improve the retrieval specificity and to ensure that we can provide multiple passages within the context window of the final question answering prompt.

Here we are chunking documents into 800 token passages with an overlap of 400 tokens.

Here we are using a simple splitter but Langchain offers more advanced splitters to reduce the chance of context being lost.

In [5]:
from langchain.document_loaders import JSONLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter


def metadata_func(record: dict, metadata: dict) -> dict:
    #Populate the metadata dictionary with keys name, summary, url, category, and updated_at.
    None

    return metadata


# For more loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/
# And 3rd party loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/#third-party-loaders
loader = JSONLoader(
    file_path="temp.json",
    jq_schema=".[]",
    content_key="content",
    metadata_func=metadata_func,
)

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
       chunk_size=800,
    chunk_overlap=400, #define chunk size and chunk overlap
)
docs = loader.load_and_split(text_splitter=text_splitter)

### Bulk Import Passages

Now that we have split each document into the chunk size of 800, we will now index data to elasticsearch using [ElasticsearchStore.from_documents](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html#langchain.vectorstores.elasticsearch.ElasticsearchStore.from_documents).

We will use Cloud ID, Password and Index name values set in the `Create cloud deployment` step.

In [11]:
from langchain_openai.llms import OpenAI

documents = vectorstore.from_documents(
    docs,
    embeddings,
    index_name="workplace-multiquery-lab",
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
)

llm = OpenAI(
    model="gpt-4o-mini",
    temperature=0,
    openai_api_key=OPENAI_API_KEY,
)
retriever = MultiQueryRetriever.from_llm(vectorstore.as_retriever(), llm)

# Question Answering with MultiQuery Retriever

Now that we have the passages stored in Elasticsearch, we can now ask a question to get the relevant passages.

In [13]:
from langchain.schema.runnable import RunnableParallel, RunnablePassthrough
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain.schema import format_document
from langchain_core.output_parsers import StrOutputParser


import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Be as verbose and educational in your response as possible. 
    
    context: {context}
    Question: "{question}"
    Answer:
    """
)

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template(
    """
---
SOURCE: {name}
{page_content}
---
"""
)


def _combine_documents(
    docs,
    document_prompt=LLM_DOCUMENT_PROMPT,
    document_separator="\n\n",
):
    doc_strings = []

    for doc in docs:
        doc_strings.append(
            document_prompt.format(
                name=doc.metadata.get("name") or "Unknown source",
                page_content=doc.page_content,
            )
        )

    return document_separator.join(doc_strings)
_context = RunnableParallel(
    context=retriever | _combine_documents,
    question=RunnablePassthrough(),
)

_context = RunnableParallel(
    context=retriever | _combine_documents,
    question=RunnablePassthrough(),
)

chain = _context | LLM_CONTEXT_PROMPT | llm | StrOutputParser()

ans = chain.invoke("what is the nasa sales team?")

print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['Alternative question 1: who are the members of the NASA sales team?  ', '    Alternative question 2: what does the NASA sales team do?  ', '    Alternative question 3: how is the NASA sales team structured?']


---- Answer ----
 The NASA sales team refers to the North America South America region within a company's sales organization. This region encompasses both North America, which includes the United States, Canada, and Mexico, as well as South America, which includes various countries in that continent. The sales team in this region is structured to effectively serve customers and achieve business objectives across these areas. 

     The NASA sales team is led by two Area Vice-Presidents: Laura Martinez oversees North America, while Gary Johnson is responsible for South America. Together, they manage a dedicated team of account managers, sales representatives, and support staff who work to identify and pursue new business opportunities, nurture existing client relationships, and ensure customer satisfaction. 

     The NASA sales team collaborates closely with other departments, such as marketing and product development, to deliver high-quality products and services to clients in both No

<br>

## 🚀 Your turn

Time to make this lab your own!


So far we've asked a single question ("what is the nasa sales team?") and looked at one answer. Now it's your turn to try the pipeline with your own questions and see how `MultiQueryRetriever` handles them.


Here's what to do:

1. **Pick 2 new questions** — Look through the sample dataset (or just get creative) and write down 2 questions you'd like the chatbot to answer, different from the example above.
2. **Ask each question** — For each question, copy the last code cell, replace the question passed to `chain.invoke(...)`, and run it.
3. **Check the logs** — Because logging is enabled, you'll see the alternative questions `MultiQueryRetriever` generated for you in the cell output. Take a look at them.
4. **Reflect** — In a markdown cell, briefly note: Did the generated questions look useful? Was the final answer correct and well grounded in the retrieved context, or did the model seem to be missing information?

<br>

💡 **Tip:**

- If an answer looks off, try asking the same question in a different way, or peek at the generated queries to see if they drifted from what you actually meant.

<br>

⭐️ **Bonus ideas:**

- Change `retriever = MultiQueryRetriever.from_llm(vectorstore.as_retriever(), llm)` to instead use `vectorstore.as_retriever()` directly (no multi-query), and compare the answers you get for the same questions. Does `MultiQueryRetriever` actually help?
- Look at the `k` parameter of `as_retriever()` (e.g. `vectorstore.as_retriever(search_kwargs={"k": 3})`) to control how many passages are retrieved per query, and see how that affects the final answer.
- Edit `LLM_CONTEXT_PROMPT` to change the tone or format of the final answer (e.g. ask for a short bullet-point summary instead of a verbose answer), and see how that changes the output.
- Ask a question that isn't covered by the sample dataset at all — does the chatbot correctly say it doesn't know, or does it make something up?

In [14]:
#1. Inspect the dataset to choose relevant questions
for record in data[:20]:
    print("Name:", record.get("name"))
    print("Summary:", record.get("summary"))
    print("---")

Name: Work From Home Policy
Summary: This policy outlines the guidelines for full-time remote work, including eligibility, equipment and resources, workspace requirements, communication expectations, performance expectations, time tracking and overtime, confidentiality and data security, health and well-being, and policy reviews and updates. Employees are encouraged to direct any questions or concerns
---
Name: April Work From Home Update
Summary: Starting May 2022, employees will need to work two days a week in the office. Coordinate with your supervisor and HR department for these days while following safety protocols.
---
Name: Wfh Policy Update May 2023
Summary: Starting May 1, 2023, our hybrid work policy will require employees to work from the office three days a week and two days remotely.
---
Name: Fy2024 Company Sales Strategy
Summary: This sales strategy document outlines objectives, focus areas, and action plans for our tech company's sales operations in fiscal year 2024. Ou

In [15]:
# Question 1
question_1 = "What steps should a new employee follow during onboarding?"

print(chain.invoke(question_1))

INFO:langchain.retrievers.multi_query:Generated queries: ['Alternative questions:  ', '    What is the onboarding process for new employees?  ', '    What are the key steps involved in onboarding a new hire?  ', '    What guidelines should new employees adhere to during their onboarding?']


 A new employee should follow several key steps during the onboarding process to ensure a smooth transition into the company. Here are the essential steps to take within the first week:

1. **Attend Orientation**: New employees should participate in an orientation session where they can meet their colleagues and learn more about the company's history, mission, and values. This is an important opportunity to understand the company culture and establish connections with fellow employees.

2. **Review Policies and Procedures**: It is crucial for new hires to familiarize themselves with the employee handbook, which contains important information about the company's policies and procedures. Reading this handbook thoroughly will help them adhere to the guidelines and understand their rights and responsibilities.

3. **Complete Required Training**: New employees may be required to attend mandatory training sessions, such as safety training or anti-harassment training. It is important to compl

In [16]:
# Question 2
question_2 = "What is the company's policy on working remotely?"

print(chain.invoke(question_2))

INFO:langchain.retrievers.multi_query:Generated queries: ['Alternative questions:  ', '    What are the guidelines for remote work at the company?  ', '    How does the company handle remote work arrangements?  ', '    What are the rules regarding telecommuting at the organization?']


 The company's policy on working remotely has evolved over time, particularly in response to the COVID-19 pandemic. Initially, in March 2020, the company implemented a full-time work-from-home policy to ensure business continuity and productivity while prioritizing employee safety. This policy allowed eligible employees to work from home full-time, provided they had the necessary approvals from their supervisors and the HR department. 

    As the situation progressed, the company adjusted its remote work policy. Starting in May 2022, a new requirement was introduced, mandating that all eligible employees work in the office at least two days a week while still allowing for three days of remote work. This adjustment aimed to balance the benefits of in-person collaboration with the flexibility of remote work.

    Most recently, effective May 1, 2023, the company has further refined its hybrid work policy. Under this updated policy, employees are now required to work from the office thre

In [17]:
#3. Inspect evidence before judging the answer
evidence = retriever.invoke(question_1)

for number, doc in enumerate(evidence, start=1):
    print(f"\n--- Passage {number} ---")
    print("Source:", doc.metadata.get("name", "Unknown source"))
    print(doc.page_content)

INFO:langchain.retrievers.multi_query:Generated queries: ['Alternative questions:  ', '    What is the onboarding process for new employees?  ', '    What are the key steps involved in onboarding a new hire?  ', '    What guidelines should new employees adhere to during their onboarding?']



--- Passage 1 ---
Source: Unknown source
As we continue to prioritize the well-being of our employees, we are making a slight adjustment to our hybrid work policy. Starting May 1, 2023, employees will be required to work from the office three days a week, with two days designated for remote work. Please communicate with your supervisor and HR department to establish your updated in-office workdays.

--- Passage 2 ---
Source: Unknown source
Starting May 2022, the company will be implementing a two-day in-office work requirement per week for all eligible employees. Please coordinate with your supervisor and HR department to schedule your in-office workdays while continuing to follow all safety protocols.

--- Passage 3 ---
Source: Unknown source
Welcome to our team! We are excited to have you on board and look forward to your valuable contributions. This onboarding guide is designed to help you get started by providing essential information about our policies, procedures, and resources.

In [18]:
#4. Add a Markdown reflection
evidence = retriever.invoke(question_1)

for number, doc in enumerate(evidence, start=1):
    print(f"\n--- Passage {number} ---")
    print("Source:", doc.metadata.get("name", "Unknown source"))
    print(doc.page_content)

INFO:langchain.retrievers.multi_query:Generated queries: ['Alternative questions:  ', '    What is the onboarding process for new employees?  ', '    What are the key steps involved in onboarding a new hire?  ', '    What guidelines should new employees adhere to during their onboarding?']



--- Passage 1 ---
Source: Unknown source
As we continue to prioritize the well-being of our employees, we are making a slight adjustment to our hybrid work policy. Starting May 1, 2023, employees will be required to work from the office three days a week, with two days designated for remote work. Please communicate with your supervisor and HR department to establish your updated in-office workdays.

--- Passage 2 ---
Source: Unknown source
Starting May 2022, the company will be implementing a two-day in-office work requirement per week for all eligible employees. Please coordinate with your supervisor and HR department to schedule your in-office workdays while continuing to follow all safety protocols.

--- Passage 3 ---
Source: Unknown source
Welcome to our team! We are excited to have you on board and look forward to your valuable contributions. This onboarding guide is designed to help you get started by providing essential information about our policies, procedures, and resources.

### Question 1
**Question:** What steps should a new employee follow during onboarding?

**Query variations:** Not yet evaluated. I need to check whether the
generated variations preserve the focus on onboarding steps for new employees.

**Grounding:** Not yet verified. I need to compare the answer with the
retrieved passages and check whether the suggested steps are explicitly
supported by the documents.

### Question 2
**Question:** What is the company's policy on working remotely?

**Query variations:** Not yet evaluated. I need to check whether the
variations focus on the company's remote-working policy without introducing
unsupported assumptions.

**Grounding:** Not yet verified. I need to check whether the retrieved
passages describe an actual policy and whether the answer adds any rules
that are absent from those passages.

In [19]:
# bonus: compare single-query and multi-query retrieval
single_retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

multi_retriever = MultiQueryRetriever.from_llm(
    single_retriever,
    llm,
)


def build_chain(selected_retriever):
    context = RunnableParallel(
        context=selected_retriever | _combine_documents,
        question=RunnablePassthrough(),
    )

    return context | LLM_CONTEXT_PROMPT | llm | StrOutputParser()


single_chain = build_chain(single_retriever)
multi_chain = build_chain(multi_retriever)

print("---- Single query ----")
print(single_chain.invoke(question_1))

print("\n---- Multi-query ----")
print(multi_chain.invoke(question_1))

---- Single query ----
 A new employee should follow several key steps during the onboarding process to ensure a smooth transition into their new role. Here are the essential steps to complete within the first week:

1. **Attend Orientation**: New employees should participate in an orientation session where they will meet their colleagues and learn about the company's history, mission, and values. This is an important opportunity to understand the organizational culture and expectations.

2. **Review Policies and Procedures**: It is crucial for new hires to familiarize themselves with the employee handbook, which contains important information about company policies and procedures. Reading this document thoroughly and adhering to the guidelines is essential for compliance and understanding workplace norms.

3. **Complete Required Training**: New employees may be required to attend mandatory training sessions, such as safety training or anti-harassment training. It is important to compl

INFO:langchain.retrievers.multi_query:Generated queries: ['Alternative questions:  ', '    What is the onboarding process for new employees?  ', '    What are the key steps involved in onboarding a new hire?  ', '    What guidelines should new employees adhere to during their onboarding?']


 During the onboarding process, a new employee should follow several key steps to ensure a smooth transition into the company. Here are the essential steps outlined in the onboarding guide:

1. **Attend Orientation**: New employees should participate in an orientation session where they will meet their colleagues and learn more about the company's history, mission, and values. This is an important opportunity to understand the company culture and establish connections with fellow team members.

2. **Review Policies and Procedures**: It is crucial for new employees to familiarize themselves with the employee handbook, which contains important information about the company's policies and procedures. Reading this handbook thoroughly will help them adhere to the guidelines and understand their rights and responsibilities.

3. **Complete Required Training**: New hires may be required to attend mandatory training sessions, such as safety training or anti-harassment training. It is important 